<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9b_SBIBM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 9b — parallel SBIBM result collector

This notebook **does not train a model and does not run an SBIBM task**. It discovers task-specific artifacts under one profile/seed tag, validates their manifests against the shared campaign contract, reports completed, failed, incompatible, and missing runs, and builds comparisons from every compatible result available so far. Re-running it therefore gives useful partial results while the remaining Colab notebooks are still running.

The ten task notebooks can be launched in independent Colab runtimes:

- [Gaussian Linear](https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9b_SBIBM_GaussianLinear.ipynb) — `gaussian_linear`
- [Gaussian Linear Uniform](https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9b_SBIBM_GaussianLinearUniform.ipynb) — `gaussian_linear_uniform`
- [Gaussian Mixture](https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9b_SBIBM_GaussianMixture.ipynb) — `gaussian_mixture`
- [Two Moons](https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9b_SBIBM_TwoMoons.ipynb) — `two_moons`
- [SLCP](https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9b_SBIBM_SLCP.ipynb) — `slcp`
- [SLCP Distractors](https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9b_SBIBM_SLCPDistractors.ipynb) — `slcp_distractors`
- [Bernoulli GLM](https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9b_SBIBM_BernoulliGLM.ipynb) — `bernoulli_glm`
- [Bernoulli GLM Raw](https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9b_SBIBM_BernoulliGLMRaw.ipynb) — `bernoulli_glm_raw`
- [SIR](https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9b_SBIBM_SIR.ipynb) — `sir`
- [Lotka--Volterra](https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/Exercise_9b_SBIBM_LotkaVolterra.ipynb) — `lotka_volterra`

## One shared 9a-matched training contract

All non-smoke task notebooks import the same configuration module and enforce:

- the fixed **10,000-simulation** bank used in Exercise 9a;
- four members for each joint $q_\phi(\theta\mid x)$ and
  $q_\eta(x\mid\theta)$ flow;
- the original Exercise-9 conditional RQS topology: 10 transformations,
  512 hidden units, four hidden layers, 16 bins, tail bound 5, no dropout;
- ten independent plain three-logit classifiers, each with four 1024-unit ReLU
  layers and no dropout, weight decay, normalization layer, residual block,
  output bound, or auxiliary objective;
- flow and classifier training for 250 epochs with batch-size row budget 32,
  Adam learning rate $10^{-4}$ divided by ten every 40 epochs down to
  $10^{-9}$, full 250-epoch patience, and held-out NLL/CE checkpoint selection;
- pure equal-prior multiclass CE as the complete classifier loss;
- arithmetic averaging of member-wise positive softmax quotients;
- normalization only as a post-training correction/check and the bridge only as
  a post-training consistency check.

The sole SBIBM-specific numerical adaptation is a fixed training-column
standardization of classifier inputs because the ten tasks use very different
coordinate scales. It is not density-aware and does not use analytic task
information.

In [ ]:
# Lightweight Colab setup: source is ephemeral; artifacts stay on Drive.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    REPO_DIR = Path("/content/nsbi-lhc-toolkit")
    if not (REPO_DIR / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, str(REPO_DIR),
        ], check=True, env=clone_env)
    subprocess.run([
        "git", "-C", str(REPO_DIR), "sparse-checkout", "set",
        "src", "workshops/ml4hep_tifr_colab",
    ], check=True)
    TUTORIAL_DIR = REPO_DIR / "workshops" / "ml4hep_tifr_colab"
    default_artifact_root = Path(
        "/content/drive/MyDrive/hybrid_nsbi_ml/exercise_9b_SBIBM"
    )
else:
    candidates = [Path.cwd(), Path.cwd() / "workshops" / "ml4hep_tifr_colab"]
    TUTORIAL_DIR = next(
        (candidate for candidate in candidates
         if (candidate / "utils_exercise9b_contract.py").exists()),
        None,
    )
    if TUTORIAL_DIR is None:
        raise FileNotFoundError(
            "Run from the repository root or workshops/ml4hep_tifr_colab."
        )
    default_artifact_root = Path.cwd() / "exercise_9b_SBIBM_artifacts"

for import_dir in (TUTORIAL_DIR, TUTORIAL_DIR.parents[1] / "src"):
    value = str(import_dir.resolve())
    if value not in sys.path:
        sys.path.insert(0, value)

ARTIFACT_ROOT = Path(
    os.environ.get("EX9B_ARTIFACT_ROOT", str(default_artifact_root))
).expanduser().resolve()
print("Reading Exercise-9b artifacts from:", ARTIFACT_ROOT)

In [ ]:
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from utils_exercise9b_contract import (
    ALL_TASKS,
    CAMPAIGN_SCHEMA,
    DEFAULT_SEED,
    METHOD_HYBRID,
    METHOD_JANA,
    METHOD_LABELS,
    METHODS,
    METRIC_SCHEMA,
    PROFILES,
    TASK_NOTEBOOKS,
    TASK_SHORT_LABELS,
    TASK_TITLES,
    aggregate_run_tag,
    campaign_run_tag,
    campaign_signature,
    expected_status,
    normalize_profile,
    validate_seed,
)
from utils_plotting import export_standalone_figure_script

PROFILE = normalize_profile(os.environ.get("EX9B_PROFILE", "PAPER"))
aggregate_seed_text = os.environ.get("EX9B_AGGREGATE_SEEDS", "").strip()
if aggregate_seed_text:
    AGGREGATE_SEEDS = tuple(dict.fromkeys(
        validate_seed(value.strip()) for value in aggregate_seed_text.split(",")
        if value.strip()
    ))
else:
    AGGREGATE_SEEDS = (validate_seed(os.environ.get("EX9B_SEED", DEFAULT_SEED)),)

campaign = PROFILES[PROFILE]
CAMPAIGN_SIGNATURE = campaign_signature(PROFILE)
AGGREGATE_RUN_TAGS = tuple(
    campaign_run_tag(PROFILE, seed) for seed in AGGREGATE_SEEDS
)
AGGREGATE_TAG = aggregate_run_tag(PROFILE, AGGREGATE_SEEDS)
RESULT_ROOT = ARTIFACT_ROOT / "results"
FIGURE_ROOT = ARTIFACT_ROOT / "figures_scripts"
AGGREGATE_DIR = FIGURE_ROOT / "aggregate" / AGGREGATE_TAG
AGGREGATE_DIR.mkdir(parents=True, exist_ok=True)

METHOD_COLORS = {METHOD_JANA: "C0", METHOD_HYBRID: "C1"}
METHOD_MARKERS = {METHOD_JANA: "o", METHOD_HYBRID: "D"}

def export_figure(fig, stem):
    script = export_standalone_figure_script(
        fig, script_name=stem, output_dir=AGGREGATE_DIR
    )
    fig.savefig(AGGREGATE_DIR / f"{stem}.png", dpi=220, bbox_inches="tight")
    fig.savefig(AGGREGATE_DIR / f"{stem}.pdf", bbox_inches="tight")
    print("Exported:", script)

print(json.dumps({
    "profile": PROFILE,
    "aggregate_seeds": AGGREGATE_SEEDS,
    "aggregate_run_tags": AGGREGATE_RUN_TAGS,
    "aggregate_tag": AGGREGATE_TAG,
    "campaign_signature": CAMPAIGN_SIGNATURE,
    "expected_tasks_per_seed": len(ALL_TASKS),
    "artifact_root": str(ARTIFACT_ROOT),
}, indent=2))

## Tagged artifact discovery

A result is included only when its status JSON and paired metric CSV exist and every contract field matches the selected profile, seed, tag, schema, ensemble sizes, epoch counts, objectives, and post-training roles. Missing and failed tasks remain visible and are never silently averaged.

In [ ]:
status_rows = []
completed_metric_paths = []
for seed in AGGREGATE_SEEDS:
    run_tag = campaign_run_tag(PROFILE, seed)
    for task in ALL_TASKS:
        expected = expected_status(task, PROFILE, seed)
        status_path = RESULT_ROOT / f"{task}__{run_tag}__status.json"
        metric_path = RESULT_ROOT / f"{task}__{run_tag}__metrics.csv"
        row = {
            "task": task,
            "task_title": TASK_TITLES[task],
            "seed": seed,
            "run_tag": run_tag,
            "notebook": TASK_NOTEBOOKS[task],
            "status": "missing",
            "message": "task notebook has not written a status manifest",
            "status_path": str(status_path),
            "metric_path": str(metric_path),
        }
        if status_path.exists():
            try:
                observed = json.loads(status_path.read_text(encoding="utf-8"))
            except Exception as exc:
                row.update(status="invalid_json", message=str(exc))
            else:
                mismatches = {
                    key: {"observed": observed.get(key), "expected": value}
                    for key, value in expected.items()
                    if observed.get(key) != value
                }
                if mismatches:
                    row.update(
                        status="incompatible",
                        message=json.dumps(mismatches, sort_keys=True),
                    )
                elif observed.get("status") == "failed":
                    row.update(
                        status="failed",
                        message=(
                            f"{observed.get('exception_type', 'Exception')}: "
                            f"{observed.get('message', '')}"
                        ),
                    )
                elif observed.get("status") != "completed":
                    row.update(
                        status="incomplete",
                        message=f"manifest status={observed.get('status')!r}",
                    )
                elif not metric_path.exists():
                    row.update(
                        status="incomplete",
                        message="completed manifest exists but metric CSV is missing",
                    )
                else:
                    row.update(status="completed", message="compatible paired metrics found")
                    completed_metric_paths.append((task, seed, run_tag, metric_path))
        status_rows.append(row)

status_table = pd.DataFrame(status_rows)
status_counts = status_table.groupby("status", as_index=False).size()
display(status_counts.style.hide(axis="index"))
display(
    status_table[["task_title", "seed", "status", "message", "notebook"]]
    .style.map(
        lambda value: (
            "background-color:#d9ead3" if value == "completed"
            else "background-color:#f4cccc" if value == "failed"
            else "background-color:#fff2cc" if value in {"incompatible", "incomplete", "invalid_json"}
            else ""
        )
    ).hide(axis="index")
)
print(
    "Completed compatible task--seed pairs:", len(completed_metric_paths),
    "/", len(ALL_TASKS) * len(AGGREGATE_SEEDS),
)

In [ ]:
required_columns = {
    "task", "profile", "seed", "run_tag", "campaign_schema",
    "campaign_signature", "metric_schema", "method", "method_label",
    "num_simulations", "flow_members", "classifier_members",
    "num_observation", "posterior_MMD", "predictive_x_MMD",
    "predictive_joint_MMD", "posterior_C2ST", "predictive_x_C2ST",
    "predictive_joint_C2ST", "classifier_objective",
    "checkpoint_objective", "normalization_role", "uses_ce_correction",
}
metric_frames = []
for task, seed, run_tag, path in completed_metric_paths:
    frame = pd.read_csv(path)
    if frame.empty or not required_columns.issubset(frame.columns):
        raise RuntimeError(f"Incomplete paired metric schema: {path}")
    expected_common = {
        "task": {task},
        "profile": {PROFILE},
        "seed": {int(seed)},
        "run_tag": {run_tag},
        "campaign_schema": {CAMPAIGN_SCHEMA},
        "campaign_signature": {CAMPAIGN_SIGNATURE},
        "metric_schema": {METRIC_SCHEMA},
        "num_simulations": {int(campaign["num_simulations"])},
        "flow_members": {int(campaign["flow_members"])},
        "classifier_members": {int(campaign["classifier_members"])},
        "method": set(METHODS),
    }
    observed_common = {
        "task": set(frame["task"].astype(str)),
        "profile": set(frame["profile"].astype(str)),
        "seed": set(frame["seed"].astype(int)),
        "run_tag": set(frame["run_tag"].astype(str)),
        "campaign_schema": set(frame["campaign_schema"].astype(str)),
        "campaign_signature": set(frame["campaign_signature"].astype(str)),
        "metric_schema": set(frame["metric_schema"].astype(str)),
        "num_simulations": set(frame["num_simulations"].astype(int)),
        "flow_members": set(frame["flow_members"].astype(int)),
        "classifier_members": set(frame["classifier_members"].astype(int)),
        "method": set(frame["method"].astype(str)),
    }
    if observed_common != expected_common:
        raise RuntimeError(
            f"Metric configuration mismatch for {path}: "
            f"observed={observed_common}, expected={expected_common}"
        )
    expected_observations = set(map(int, campaign["observations"]))
    for method in METHODS:
        rows = frame.loc[frame["method"] == method]
        if (
            set(rows["num_observation"].astype(int)) != expected_observations
            or rows["num_observation"].astype(int).duplicated().any()
        ):
            raise RuntimeError(f"Incomplete or duplicated {method} rows in {path}")
    metric_frames.append(frame)

if metric_frames:
    aggregate_results = pd.concat(metric_frames, ignore_index=True)
    keys = ["task", "seed", "num_observation", "method"]
    if aggregate_results.duplicated(keys).any():
        raise RuntimeError("Duplicate task/seed/observation/method rows detected")
    summary = (
        aggregate_results.groupby(["task", "method", "method_label"], as_index=False)
        .agg(
            mean_posterior_MMD=("posterior_MMD", "mean"),
            std_posterior_MMD=("posterior_MMD", "std"),
            mean_predictive_x_MMD=("predictive_x_MMD", "mean"),
            std_predictive_x_MMD=("predictive_x_MMD", "std"),
            mean_posterior_C2ST=("posterior_C2ST", "mean"),
            mean_predictive_x_C2ST=("predictive_x_C2ST", "mean"),
            observations=("num_observation", "nunique"),
            independent_seeds=("seed", "nunique"),
        )
    )
    paired = aggregate_results.pivot(
        index=["task", "seed", "num_observation"],
        columns="method",
        values=[
            "posterior_MMD", "predictive_x_MMD", "predictive_joint_MMD",
            "posterior_C2ST", "predictive_x_C2ST", "predictive_joint_C2ST",
        ],
    )
    paired.columns = [f"{metric}__{method}" for metric, method in paired.columns]
    paired = paired.reset_index()
    epsilon = 1.0e-12
    for metric in ["posterior_MMD", "predictive_x_MMD", "predictive_joint_MMD"]:
        paired[f"log10_hybrid_over_jana__{metric}"] = np.log10(
            (paired[f"{metric}__{METHOD_HYBRID}"] + epsilon)
            / (paired[f"{metric}__{METHOD_JANA}"] + epsilon)
        )
    aggregate_results.to_csv(AGGREGATE_DIR / "available_tasks_metrics_long.csv", index=False)
    summary.to_csv(AGGREGATE_DIR / "available_tasks_summary_by_method.csv", index=False)
    paired.to_csv(AGGREGATE_DIR / "available_tasks_paired_comparison.csv", index=False)
    display(summary.style.format(precision=4).hide(axis="index"))
else:
    aggregate_results = pd.DataFrame()
    summary = pd.DataFrame()
    paired = pd.DataFrame()
    print("No compatible completed task metrics are available for this tag yet.")

missing_pairs = [
    [row.task, int(row.seed)]
    for row in status_table.itertuples()
    if row.status != "completed"
]
(AGGREGATE_DIR / "aggregate_manifest.json").write_text(json.dumps({
    "profile": PROFILE,
    "aggregate_seeds": AGGREGATE_SEEDS,
    "aggregate_run_tags": AGGREGATE_RUN_TAGS,
    "campaign_schema": CAMPAIGN_SCHEMA,
    "campaign_signature": CAMPAIGN_SIGNATURE,
    "metric_schema": METRIC_SCHEMA,
    "available_task_seed_pairs": len(completed_metric_paths),
    "expected_task_seed_pairs": len(ALL_TASKS) * len(AGGREGATE_SEEDS),
    "complete": not missing_pairs,
    "missing_or_invalid_task_seed_pairs": missing_pairs,
}, indent=2), encoding="utf-8")

## Partial or complete JANA-versus-hybrid comparisons

Every panel uses only compatible completed pairs. Gray crosses mark tasks with no result under the selected tag. Negative values in the log-ratio plot mean that the CE correction reduced MMD relative to the direct JANA flow using the same frozen flow ensemble.

In [ ]:
if not aggregate_results.empty:
    x_positions = np.arange(len(ALL_TASKS))
    labels = [TASK_SHORT_LABELS[task] for task in ALL_TASKS]
    offsets = {METHOD_JANA: -0.16, METHOD_HYBRID: 0.16}

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.0), constrained_layout=True)
    for x_index, task in enumerate(ALL_TASKS):
        task_rows = aggregate_results.loc[aggregate_results["task"] == task]
        if task_rows.empty:
            for ax in axes:
                ax.scatter([x_index], [1.0e-7], marker="x", color="0.70")
            continue
        for ax, metric in [(axes[0], "posterior_MMD"), (axes[1], "predictive_x_MMD")]:
            for method in METHODS:
                rows = task_rows.loc[task_rows["method"] == method]
                jitter = np.linspace(-0.05, 0.05, len(rows)) if len(rows) > 1 else np.zeros(1)
                values = np.maximum(rows[metric].to_numpy(), 1.0e-7)
                ax.scatter(
                    x_index + offsets[method] + jitter, values, s=22,
                    color=METHOD_COLORS[method], marker=METHOD_MARKERS[method], alpha=0.48,
                )
                ax.scatter(
                    [x_index + offsets[method]], [max(rows[metric].mean(), 1.0e-7)],
                    s=70, color=METHOD_COLORS[method], marker=METHOD_MARKERS[method],
                    edgecolor="black", zorder=4, label=METHOD_LABELS[method],
                )
    axes[0].set(yscale="log", ylabel="Gaussian-kernel MMD", title="(a) Posterior MMD")
    axes[1].set(yscale="log", ylabel="Gaussian-kernel MMD", title="(b) Posterior-predictive $x$ MMD")
    for ax in axes:
        ax.set_xticks(x_positions, labels, rotation=35, ha="right")
        ax.grid(axis="y", alpha=0.25)
    handles, legend_labels = axes[0].get_legend_handles_labels()
    unique = dict(zip(legend_labels, handles))
    axes[0].legend(unique.values(), unique.keys(), fontsize=8)
    export_figure(fig, "available_tasks_jana_vs_hybrid_figure5_mmd")
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.8), constrained_layout=True)
    for x_index, task in enumerate(ALL_TASKS):
        rows = paired.loc[paired["task"] == task]
        for ax, metric in [(axes[0], "posterior_MMD"), (axes[1], "predictive_x_MMD")]:
            column = f"log10_hybrid_over_jana__{metric}"
            if rows.empty:
                ax.scatter([x_index], [0.0], marker="x", color="0.70")
                continue
            jitter = np.linspace(-0.10, 0.10, len(rows)) if len(rows) > 1 else np.zeros(1)
            ax.scatter(x_index + jitter, rows[column], s=24, color="C3", alpha=0.50)
            ax.scatter([x_index], [rows[column].mean()], s=70, marker="D", color="C3", edgecolor="black")
    axes[0].set(title="(a) Posterior MMD ratio")
    axes[1].set(title="(b) Predictive-$x$ MMD ratio")
    for ax in axes:
        ax.axhline(0.0, color="black", ls="--", lw=1)
        ax.set(
            ylabel=r"$\log_{10}(\mathrm{MMD}_{\rm hybrid}/\mathrm{MMD}_{\rm JANA})$"
        )
        ax.set_xticks(x_positions, labels, rotation=35, ha="right")
        ax.grid(axis="y", alpha=0.25)
    export_figure(fig, "available_tasks_paired_mmd_log_ratio")
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.0), constrained_layout=True)
    for x_index, task in enumerate(ALL_TASKS):
        task_rows = aggregate_results.loc[aggregate_results["task"] == task]
        for ax, metric in [(axes[0], "posterior_C2ST"), (axes[1], "predictive_x_C2ST")]:
            for method in METHODS:
                rows = task_rows.loc[task_rows["method"] == method]
                if rows.empty:
                    continue
                jitter = np.linspace(-0.05, 0.05, len(rows)) if len(rows) > 1 else np.zeros(1)
                ax.scatter(
                    x_index + offsets[method] + jitter, rows[metric], s=22,
                    color=METHOD_COLORS[method], marker=METHOD_MARKERS[method], alpha=0.50,
                )
                ax.scatter(
                    [x_index + offsets[method]], [rows[metric].mean()], s=70,
                    color=METHOD_COLORS[method], marker=METHOD_MARKERS[method],
                    edgecolor="black", zorder=4,
                )
    axes[0].set(ylabel="C2ST", title="(a) Posterior C2ST")
    axes[1].set(ylabel="C2ST", title="(b) Posterior-predictive $x$ C2ST")
    for ax in axes:
        ax.axhline(0.5, color="black", ls="--", lw=1)
        ax.set_xticks(x_positions, labels, rotation=35, ha="right")
        ax.grid(axis="y", alpha=0.25)
    export_figure(fig, "available_tasks_supplemental_c2st")
    plt.show()

## Run/re-run pattern

Launch any subset of the task notebooks in parallel with the same `PROFILE` and `BASE_SEED`. Re-run this collector whenever one finishes. A changed training contract produces a different tag and is therefore excluded automatically rather than mixed with earlier artifacts.